In [ ]:
import base64
import json
import numpy as np
import zlib

from PIL import Image
from dataclasses import dataclass
from typing import Optional, Tuple

In [ ]:
def array_hash(x: np.ndarray):
    x = x.astype(np.uint32)
    x ^= x >> 16
    x *= 0x7feb352d
    x ^= x >> 15
    x *= 0x846ca68b
    x ^= x >> 16
    return x

In [ ]:
@dataclass
class Window:
    """Pair of (row, col) tuples that define an inclusive-exclusive 2D range"""
    
    v0: Tuple[int, int]
    v1: Tuple[int, int]
    
    def __len__(self):
        return self.span(0) * self.span(1)
        
    def span(self, dim):
        return self.v1[dim] - self.v0[dim]

In [ ]:
@dataclass
class TileMap:
    """Defines the site point and voxel assignments for a set of tiles"""
    
    index: np.ndarray
    seeds: np.ndarray
    sites: np.ndarray
    tiles: np.ndarray
        
    def mask(self, tile=None):
        if tile is None:
            return self.tiles < self.sites.shape[0]
        else:
            assert tile < self.site.shapes[0]
            return self.tiles == tile
        
    def to_json(self):
        return {
            "index": self.index.tolist(),
            "seeds": self.seeds.tolist(),
            "sites": self.sites.tolist(),
            "tiles": self.tiles.tolist(),
        }
    
    def from_json(self, json):
        self.index = np.array(json["index"])
        self.seeds = np.array(json["seeds"])
        self.sites = np.array(json["sites"])
        self.tiles = np.array(json["tiles"])

In [ ]:
TILE_PERIOD = 64
TILE_RESOLUTION = 16
TILE_JITTER = 6

In [ ]:
def generate_seeds(window: Window):    
    return window.v0 + np.stack(
        np.meshgrid(
            np.arange(window.span(0)),
            np.arange(window.span(1)),
        ),
        axis=-1,
    )

In [ ]:
def generate_shifts(seeds: np.ndarray):
    h_x = seeds[:, :, 0] % TILE_PERIOD
    h_y = seeds[:, :, 1] % TILE_PERIOD
    h = array_hash(h_x + array_hash(h_y))
    x = 2.0 * ((h >> 1) & 0xff) / 0xff - 1.0
    y = 2.0 * ((h >> 9) & 0xff) / 0xff - 1.0
    return np.stack([x, y], axis=-1)

In [ ]:
def generate_sites(seeds, jitter=0.35):
    return TILE_RESOLUTION * (jitter * generate_shifts(seeds) + (seeds + 0.5))

In [ ]:
def generate_tiles(window: Window):
    # Generate the sites for a padded window.
    padded = Window(
        v0=(window.v0[0] - 2, window.v0[1] - 2),
        v1=(window.v1[0] + 2, window.v1[1] + 2),
    )
    
    # Generate the seeds for the padded window. Keep track of which ones were padding.
    seeds = generate_seeds(padded)
    xmask = np.logical_and(seeds[:, :, 0] >= window.v0[0], seeds[:, :, 0] < window.v1[0])
    ymask = np.logical_and(seeds[:, :, 1] >= window.v0[1], seeds[:, :, 1] < window.v1[1])
    smask = np.logical_and(xmask, ymask) 
    
    # Generate the voronoi sites. These are completely deterministic. 
    sites = generate_sites(seeds)
    
    # Generate the voronoi cells.
    # TODO: For integer points, make sure that this is deterministic. 
    tiles = voxeloo.voronoi.rasterize_2d(
        sites.reshape(-1, 2) - TILE_RESOLUTION * np.array(padded.v0),
        (
            TILE_RESOLUTION * padded.span(0),
            TILE_RESOLUTION * padded.span(1),
        )
    )
    
    # Re-index the tiles to exclude the padded sites.
    n = len(window)
    index = n * np.ones(len(padded), dtype=int)
    index[smask.flatten()] = np.arange(n)
    tiles = index[tiles]
    
    return TileMap(
        index=np.arange(n).reshape(window.span(1), window.span(0)),
        seeds=seeds[smask, :],
        sites=sites[smask, :],
        tiles=tiles,
    )

### Generate the tile map

In [ ]:
%%time

tm = generate_tiles(
    Window(
        v0=(0, 0),
        v1=(64, 64),
    )
)

### Plot the tile map

In [ ]:
%%time

palette = np.random.randint(0, 255, size=(tm.tiles.max() + 1, 3), dtype=np.uint8)
palette[-1, :] = [0, 0, 0]
Image.fromarray(palette[tm.tiles])

### Save the tile map

In [ ]:
data = zlib.compress(json.dumps(tm.to_json()).encode("utf-8"))
with open("data/tile_map", "wb") as f:
    f.write(data)